### The Cloud Configuration
- AI API Key
- Cloud Service Key

In [ ]:
import os
import requests
# cloud project settings
data_set = "mta_dev"
gcs_bucket = "ozkary_data_lake_ozkary-de-101"
project_id = "ozkary-de-101"
region = "us-east1"

# set the env variable for the api key  ONLY WHEN NOT USING VERTEX AI
api_key_file = os.environ.get("GOOGLE_API_KEY")
api_key = ""
with open(api_key_file, 'r') as f:
    api_key = f.read().strip()

print(api_key_file)

# validate the service account key
sa_key = os.environ["GOOGLE_APPLICATION_CREDENTIALS"]
print(sa_key)

/home/ozkary/.gcp/gemini.key
/home/ozkary/.gcp/ozkary-de-101.json


#### Review your service account profile context

In [3]:
from google.cloud import storage
from google.cloud import bigquery
import fnmatch

def verify_gcs_access():
    client = storage.Client()
    # This tests 'serviceusage.serviceUsageConsumer' and 'storage.objectViewer'
    buckets = list(client.list_buckets(max_results=1))
    print(f"✅ Storage connection: {client.project}")
    print(f"📁 Found access to {len(buckets)} buckets.")
    return buckets

def verify_bq_access():
    client = bigquery.Client()
    # Test 'metadataViewer' by listing datasets
    datasets = list(client.list_datasets())
    print(f"✅ BQ Connection Verified: {client.project}")
    print(f"📁 Found {len(datasets)} datasets (e.g., {[d.dataset_id for d in datasets[:3]]})")


def list_gcs_files(bucket_name: str, pattern: str = "*") -> dict:
    
    try:
        client = storage.Client()
        blobs = client.list_blobs(bucket_name)
        file_names = [blob.name for blob in blobs]
        
        # Apply pattern matching locally
        matched_files = fnmatch.filter(file_names, pattern)
        
        return {
            "status": "success",
            "bucket": bucket_name,
            "pattern": pattern,
            "files": matched_files if matched_files else "No matching files found."
        }
    except Exception as e:
        return {"status": "error", "message": str(e)}

# verify_gcs_access()
verify_bq_access()
# print(gcs_bucket)
list_gcs_files(gcs_bucket)

✅ BQ Connection Verified: ozkary-de-101
📁 Found 3 datasets (e.g., ['mta_data', 'mta_dev', 'trips_data'])


{'status': 'success',
 'bucket': 'ozkary_data_lake_ozkary-de-101',
 'pattern': '*',
 'files': ['turnstile/',
  'turnstile/240106.csv.gz',
  'turnstile/240113.csv.gz',
  'turnstile/240120.csv.gz',
  'turnstile/240127.csv.gz',
  'turnstile/240203.csv.gz',
  'turnstile/240210.csv.gz',
  'turnstile/240217.csv.gz',
  'turnstile/240302.csv.gz',
  'turnstile/240309.csv.gz',
  'turnstile/240316.csv.gz',
  'turnstile/240323.csv.gz']}

### Agents Governance (The "How")  System Prompt
In this cell, you define a variable that acts as the "Specifications" for your agents. 

In [5]:
# Governance & Naming Standards

GOVERNANCE_RULES = f"""
- Dataset: Use '{data_set}' for ALL tables, views, and procedures.
- Field Naming: Use snake_case for all column names with lower case letters (e.g., station_name, turnstile_id).
- External Table Naming: Use 'ext_' prefix (e.g.  ext_turnstile)
- Table Naming: Use 'dim_' prefix for physical tables (e.g., dim_turnstile).
- View Naming: Use 'vw_' prefix for logic layers (e.g., vw_turnstile).
- Stored Procedures Naming:  Use 'sp_' prefix with the area name and action name (e.g., sp_station_incremental_update) 
- Lineage: Every 'CREATE' statement must include a description identifying the source GCS path.
- Data Pipeline Naming: Use 'dp_' prefix with the area name and action (e.g., dp_station_incremental_update)
"""

PROJECT_REQUIREMENTS = """
Analyze the MTA dataset and identify all dimension areas (e.g., Station, Booth, Time).
- Create a view for each dimension: vw_{area}. 
- Each view should generate a surrogate key for the record using a combination of unique fields 'sk_{area}' use using TO_HEX(SHA1(CONCAT(natural_keys))) 
- Create a central fact view: vw_turnstile.
- The Facts must use these 'sk_' keys as Foreign Keys to join with Dimensions
- Combine the 'date' and 'time' fields from the raw source into a single 'created' DATETIME field.
- Ensure type casting is done on the views (e.g., strings to int)
- The final output should follow a Star Schema architecture to simplify BI reporting.
"""


# Agents


### Setup & Tool Initialization
First, we install the library and initialize the MCP Toolsets. These act as the "drivers" for our agents.

- pip install google-adk

In [6]:
import aiohttp

async def check_managed_server(toolset):
    """helper function to validate the tool"""
    try:
        # This is the "Pulse Check" for MCP
        tools = await toolset.get_tools()

        if tools:
            print(f"✅ Connection Validated!")
            print(f"📡 Found {len(tools)} tools.")
            # Print the first few to see what the Agent will have access to
            print(f"🛠️  Available tools: {[t.name for t in tools[:3]]}...")
        else:
            print("⚠️  Connected, but no tools were returned. Check IAM roles.")

    except Exception as e:
        print(f"❌ Connection Failed: {e}")
        # If this fails with 401/403, our Bearer token or Project ID header is the culprit.



#### Connect the MCP tools
Initialize and inspect each tool

In [8]:
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from mcp import StdioServerParameters

# The ADK will execute 'python gcs_toolset.py' in the background
gcs_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="python3",
            args=["./tools/gcs_toolset.py"], # This triggers the 'main' in your file
            env={
                "GOOGLE_APPLICATION_CREDENTIALS": sa_key,
                "GOOGLE_CLOUD_PROJECT": project_id
            }
        )
    )
)

await check_managed_server(gcs_toolset)

✅ Connection Validated!
📡 Found 2 tools.
🛠️  Available tools: ['list_mta_files', 'read_file_preview']...


In [11]:
from google import auth
from google.oauth2 import service_account
from google.adk.agents.llm_agent import Agent, LlmAgent
from google.adk.models import Gemini
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import SseConnectionParams, StreamableHTTPConnectionParams

print('Validating the account context')
auth.default()

# set the scope for the tool identity
SCOPES = ["https://www.googleapis.com/auth/cloud-platform"]
creds = service_account.Credentials.from_service_account_file(
    sa_key, 
    scopes=SCOPES
)

# Force refresh the token
request = auth.transport.requests.Request()
creds.refresh(request)

# This ensures BQ/GCS knows who you are and who to bill
headers = {
    "Authorization": f"Bearer {creds.token}",
    "x-goog-user-project": project_id,
    "Content-Type": "application/json"
}

print("Token expires:", creds.expiry)

Validating the account context
Token expires: 2026-03-31 00:48:12.887888


In [13]:

# Initialize BQ with direct credentials
from google.adk.tools.bigquery import BigQueryToolset, BigQueryCredentialsConfig
from google.adk.tools.bigquery.config import BigQueryToolConfig, WriteMode
from google.oauth2 import service_account


# Configure the credentials (using your existing 'creds' object)
bq_config = BigQueryCredentialsConfig(credentials=creds)

# WriteMode.ALLOWED allows the agent to run CREATE TABLE, INSERT, etc.
tool_config = BigQueryToolConfig(write_mode=WriteMode.ALLOWED)

# 3. Initialize the first-party toolset
bq_toolset = BigQueryToolset(
    credentials_config=bq_config,
    bigquery_tool_config=tool_config
)

print("✅ Local MCP Toolsets initialized (No Remote Gateway needed).")

✅ Local MCP Toolsets initialized (No Remote Gateway needed).


In [15]:

await check_managed_server(bq_toolset)

✅ Connection Validated!
📡 Found 11 tools.
🛠️  Available tools: ['get_dataset_info', 'get_table_info', 'list_dataset_ids']...


### Define the LLM Model

In [22]:
import importlib
import tools.agent_runner

# reload the library
importlib.reload(tools.agent_runner)
from tools.agent_runner import run_agent_task
# set the model information
llm_model = "gemini-2.5-flash"

# Use Vertex AI instead, authenticated with the service account
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = region

print("Connectors initialized: GCS (Data Lake) and BigQuery (Warehouse).")

Connectors initialized: GCS (Data Lake) and BigQuery (Warehouse).


### The Ingestion Agent (GCS Discovery)
This agent acts as your "scout." It finds the raw turnstile data so you don't have to hardcode paths.

In [23]:

# Find the GCS pattern
ingestion_agent = Agent(
    name="StorageIngestor",
    model=llm_model,
    instruction=f"""Identify the URI pattern for the MTA turnstile .gz files.
    If you find multiple files like 'YYMMDD.csv.gz' as '240106.csv.gz',
    return the wildcard pattern that can enable us to create an external table on BigQuery: 'gs://{gcs_bucket}/'.""",
    tools=[gcs_toolset]
)

gcs_pattern = await run_agent_task(ingestion_agent,"What is the GCS pattern for the MTA turnstile gzipped files?")
print(f"Pattern Discovered: {gcs_pattern}")

--- [TASK START]: StorageIngestor ---


[StorageIngestor]: No response captured.
[StorageIngestor]: No response captured.
[StorageIngestor]: No response captured.The GCS pattern for the MTA turnstile gzipped files is: `gs://ozkary_data_lake_ozkary-de-101/turnstile/*.csv.gz`
--- [TASK COMPLETE]: StorageIngestor ---

Pattern Discovered: No response captured.The GCS pattern for the MTA turnstile gzipped files is: `gs://ozkary_data_lake_ozkary-de-101/turnstile/*.csv.gz`


### The Warehouse Agent (Schema & Modeling)
This agent handles the heavy lifting: creating the External Table to read from GCS and the Deduplicated View.

In [24]:
# Define the database agent
warehouse_agent = Agent(
    name="BigQueryArchitect",
    model=llm_model,
    instruction=f"""You are a BigQuery Architect. STRICKLY FOLLOW these Governance Rules: {GOVERNANCE_RULES}
    - First, create an EXTERNAL TABLE using the {gcs_bucket} folder as the name and the URI pattern: {gcs_pattern}.
    - Set the source_format to 'CSV' and set compression to 'GZIP'.
    - Once created, run 'GET_TABLE_INFO' to confirm the schema.""",
    tools=[bq_toolset]
)

In [ ]:
#  Create the External Table
user_prompt = "Create the external table for the MTA gzipped files now."
resp = await run_agent_task(warehouse_agent, user_prompt)
print(resp)

--- [TASK START]: BigQueryArchitect ---
[BigQueryArchitect]: The external table `ext_turnstile` has been successfully created with the following schema:

- `control_area` (STRING)
- `unit` (STRING)
- `scp` (STRING)
- `station_name` (STRING)
- `line_name` (STRING)
- `division` (STRING)
- `turnstile_date` (STRING)
- `turnstile_time` (STRING)
- `description` (STRING)
- `entries` (INTEGER)
- `exits` (INTEGER)

The data source is `gs://ozkary_data_lake_ozkary-de-101/turnstile/*.csv.gz`, with `CSV` format and `GZIP` compression.
--- [TASK COMPLETE]: BigQueryArchitect ---

The external table `ext_turnstile` has been successfully created with the following schema:

- `control_area` (STRING)
- `unit` (STRING)
- `scp` (STRING)
- `station_name` (STRING)
- `line_name` (STRING)
- `division` (STRING)
- `turnstile_date` (STRING)
- `turnstile_time` (STRING)
- `description` (STRING)
- `entries` (INTEGER)
- `exits` (INTEGER)

The data source is `gs://ozkary_data_lake_ozkary-de-101/turnstile/*.csv.gz`, w

### Dimensional Modeling

In [21]:
# We reuse 'warehouse_agent' because it already knows about the external table

star_schema_prompt = f"""
Now that the external table exists, perform the following architectural tasks:
- ANALYZE the external table schema.
- Propose the Start Schema Views for each area
- FOLLOW GOVERNANCE: follow the same governance previously provided
- EXECUTE REQUIREMENTS: {PROJECT_REQUIREMENTS}
"""


resp = await run_agent_task(warehouse_agent, star_schema_prompt)
print(resp)

--- [TASK START]: BigQueryArchitect ---
[BigQueryArchitect]: The star schema views have been successfully created in the `mta_dev` dataset, following all the provided governance rules.

Here's a summary of the created views:

**Dimension Views:**

*   **`vw_station`**: Contains unique station information (`station_name`, `line_name`, `division`) with a generated `sk_station` surrogate key.
*   **`vw_booth`**: Contains unique booth identification (`control_area`, `unit`, `scp`) with a generated `sk_booth` surrogate key.
*   **`vw_date`**: Provides a comprehensive date dimension with `full_date`, `year`, `month`, `day`, `day_of_week`, `day_name`, `week_of_year`, `quarter`, and a generated `sk_date` surrogate key.
*   **`vw_time`**: Offers a time dimension with `full_time`, `hour`, `minute`, and a generated `sk_time` surrogate key.
*   **`vw_entry_type`**: Stores unique entry descriptions (`entry_description`) with a generated `sk_entry_type` surrogate key.

**Fact View:**

*   **`vw_turn

### The Physical Model Process

In [27]:
# Physical Schema Design Review
design_schema_prompt = """Continue to use the same governance and requirements to propose a physical model for our data. 
Start by creating the physical table for the station dimension. This table should include all the fields from its source view station.
Query the catalog information to find out the schema for the view. Track lineage from source to target."""
resp = await run_agent_task(warehouse_agent, design_schema_prompt)
print(resp)


--- [TASK START]: BigQueryArchitect ---
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.The physical table `dim_station` has been successfully created in the `mta_dev` dataset with the following schema:

*   **sk_station**: STRING
*   **station_name**: STRING
*   **line_name**: STRING
*   **division**: STRING

The table includes the lineage description: "Physical dimension table for station data. Lineage: Created from `ozkary-de-101.mta_dev.vw_station` which sources data from `gs://ozkary_data_lake_ozkary-de-101/turnstile/*.csv.gz`."
--- [TASK COMPLETE]: BigQueryArchitect ---

No response captured.The physical table `dim_station` has been successfully created in the `mta_dev` dataset with the following schema:

*   **sk_station**: STRING


### The Incremental Update Strategy

In [28]:
#  Incremental Loading Logic
incremental_update_design_prompt = """ Create the incremental update logic for the station entity.
    We need a process (or MERGE statements) to ensure that as data is made available on our view,
    we can merge to the dimension table. Write the stored procedure for this process.
    Follow our existing governance for any procedures created."""

resp = await run_agent_task(warehouse_agent, incremental_update_design_prompt)
print(resp)

--- [TASK START]: BigQueryArchitect ---
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchi

### Create the orchestration

In [29]:
# Create the a Pipeline
station_pipeline_design_prompt = """ 
    Create a pipeline for our station data. 
    This pipeline must contain a tasks that calls the station incremental update stored procedure."""     

resp = await run_agent_task(warehouse_agent, station_pipeline_design_prompt)
print(resp)

--- [TASK START]: BigQueryArchitect ---
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchi

### Rub the Pipeline

In [32]:
# Final Execution of the Master Pipeline
run_pipeline_prompt = """
    Run the station Pipeline
    Once complete, report the status of the run and the row counts from the table."""

resp = await run_agent_task(warehouse_agent, run_pipeline_prompt)
print(resp)

--- [TASK START]: BigQueryArchitect ---
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.The station pipeline (sp_station_incremental_update) has been executed successfully.
The `mta_dev.dim_station` table now contains 378 rows.
--- [TASK COMPLETE]: BigQueryArchitect ---

No response captured.The station pipeline (sp_station_incremental_update) has been executed successfully.
The `mta_dev.dim_station` table now contains 378 rows.


### The Tester (Verification)


In [33]:
test_prompt = "Verify the data load and show five unique station names"
resp = await run_agent_task(warehouse_agent, test_prompt)
print(resp)

--- [TASK START]: BigQueryArchitect ---
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.The data load has been verified. Here are five unique station names:
* 59 ST
* 5 AV/59 ST
* 57 ST-7 AV
* 49 ST
* TIMES SQ-42 ST
--- [TASK COMPLETE]: BigQueryArchitect ---

No response captured.The data load has been verified. Here are five unique station names:
* 59 ST
* 5 AV/59 ST
* 57 ST-7 AV
* 49 ST
* TIMES SQ-42 ST


In [36]:
review_prompt = f"Query the catalog and list tables, views and procedures that we created in our dataset {project_id}.{data_set}"
resp = await run_agent_task(warehouse_agent, review_prompt)
print(resp)

--- [TASK START]: BigQueryArchitect ---
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.
[BigQueryArchitect]: No response captured.Here are the tables and views found in the `mta_dev` dataset for the `ozkary-de-101` project:

**Tables:**
*   `dim_station`
*   `ext_turnstile`

**Views:**
*   `vw_booth`
*   `vw_date`
*   `vw_entry_type`
*   `vw_station`
*   `vw_time`
*   `vw_turnstile`

Please note that this tool cannot directly list stored procedures.
--- [TASK COMPLETE]: BigQueryArchitect ---

No response captured.Here are the tables and views found in the `mta_dev` dataset for the `ozkary-de-101` project:

**Tables:**
*   `dim_station`
*   `ext_turnstile`

**Views:**
*   `vw_booth`
*   `vw_date`
*   `vw_entry_type`
*   `vw_station`
*   `vw_time`
*   `vw_turnstile`

Please note that this tool cannot directly list stored procedures.
